# Corridas Lucrativas

Nos tempos modernos as pessoas têm feito de seu transporte os serviços de taxi associados a aplicativos onde as mesmas solicitam os serviços no aplicativo e os transportadores disponiveis no aplicativo respondem aos pedidos dependendo da sua localização e disponibilidade, este modelo de negocio permite criar um marketplace onde individuos com os seus veiculos individuais podem prestar serviços a outros individuos que precisam destes serviços de taxi, neste caso o produto comercializado são as caronas. Neste projeto o nosso objetivo é estudar o contexto da empresa Zuber que irá lançar um aplicativo de caronas em Chicago, Estados Unidos da America.

## 1 Inicialização

De forma a organizar o nosso trabalho iremos importar as bibliotecas e ferramentas que iremos usar em nosso projeto para poderem ser usadas nas proximas celulas do notebook para podermos executar as funções ja definidas nestas bibliotecas e fazer uso da reutilização do que ja foi criado.

__Carregando Bibliotecas Necessarias__

Na célula abaixo são importadas as bibliotecas que serão usadas no nosso estudo e estas são: __pandas__, __numpy__, __math__, __pyplot__, __stats__, __seaborn__, __requests__, __re__ e __bs4__.

In [ ]:
#Carregando as bibliotecas necessarias

import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
from scipy import stats as st
import seaborn as sns
import requests
import re
from bs4 import BeautifulSoup

## Carregando os dados

Nesta celula serão importados os dados de fontes externas para se fazer a analise e calculos que no fim nos darão informações acerca do mercado de taxis de aplicativos.

In [ ]:
#Minerando a pagina web e criacao de objeto BeautifulSoup

URL='https://practicum-content.s3.us-west-1.amazonaws.com/data-analyst-eng/moved_chicago_weather_2017.html'
req = requests.get(URL)
soup = BeautifulSoup(req.text, 'lxml')
sns.set_theme(style="darkgrid")

df_01 = pd.read_csv("/datasets/project_sql_result_01.csv")

df_04 = pd.read_csv("/datasets/project_sql_result_04.csv")

df_07 = pd.read_csv("/datasets/project_sql_result_07.csv")





## Analizando os dados sobre o clima em Chicago em Novembro de 2017

__Criando tabela com os dados encontrados na tabela do website__

In [ ]:
tabela = soup.find('table', attrs={'id': 'weather_records'})
nomes_colunas = []
for row in tabela.find_all('th'):
        nomes_colunas.append(row.text)
        
conteudo_tabela = []

for row in tabela.find_all('tr'):
    if not row.find_all('th'):
        conteudo_tabela.append([item.text for item in row.find_all('td')])
weather_records = pd.DataFrame(conteudo_tabela, columns = nomes_colunas)


## __Atualizando os nomes das colunas__

Nesta celula abaixo os nomes das colunas foram convertidos para nomes mais curtos e sem espaço entre as palavras de formas a manter o padrão e ser mais facil trabalhar com nomes mais simples e sem espaço.

In [ ]:
weather_records = weather_records.rename(columns={'Date and time':'data', 'Temperature':'temperatura'})

## __Atualizando os tipos de dados das Colunas__

Os campos 'data' do dataframe weather_records e 'start_ts' do dataframe df_07 armazedam dados de datas e horas ou nesse caso timestamps mas os seus tipos de dados é string então seria viável converter estes campos para o tipo de dados datetime que é mais apropriado para trabalhar com dados deste tipo. É também encontrada a coluna temperatura que será convertida para o tipo de dado float.

__Atualizando as colunas do tipo de dados string para o tipo datetime__

In [ ]:
weather_records['data'] = pd.to_datetime(weather_records['data'])

df_07['start_ts'] = pd.to_datetime(df_07['start_ts'])

print(weather_records.info())

__Atualizando a coluna temperatura para o tipo de dados float__

In [ ]:
weather_records['temperatura'] = weather_records['temperatura'].astype(float)
print(weather_records.info())


__Imprimindo as 10 primeiras linhas da tabela__

In [ ]:
print(weather_records.head(10))

__Criando categoria das datas para periodos__

In [ ]:
# weather_records[weather_records['data'].dt.isocalendar().week == 44]

def categorizar_tempo(row):
    data = row['data']
    if data.hour<=6:
        return 'madrugada'
    elif data.hour <= 11:
        return 'manha'
    elif data.hour <= 18:
        return 'tarde'
    elif data.hour <= 23:
        return 'noite'

__Grafico da temperatura durante o mês de Novembro__

In [ ]:

sns.lineplot(x="dia", y="temperatura", data=weather_records)

__Criando categorias para os periodos do dia e associando com os tipos de climas__

In [ ]:
# weather_records.pivot_table(values="Description", index="periodo", aggfunc="count")
weather_records.Description.unique()

tipo = {'broken clouds':'bom',
        'scattered clouds':'bom',
        'overcast clouds':'bom',
       'sky is clear':'bom',dd
        'mist':'bom',
        'drizzle':'mau',
        'light rain':'mau',
        'moderate rain':'mau',
       'fog':'mau',
        'light intensity drizzle':'bom',
        'few clouds':'bom',
       'thunderstorm with drizzle':'mau',
        'proximity thunderstorm':'mau',
       'proximity thunderstorm with rain':'perigo',
        'light snow':'mau', 
        'haze':'mau',
       'thunderstorm with light rain':'mau',
        'heavy intensity rain':'perigo',
       'thunderstorm with rain':'perigo'}

weather_records['tipo'] = weather_records.Description.map(tipo)
clima_perigo = weather_records.query('tipo == "perigo"').pivot_table(values="tipo", index="periodo", aggfunc="count")
clima_mau = weather_records.query('tipo == "mau"').pivot_table(values="tipo", index="periodo", aggfunc="count")
clima_bom = weather_records.query('tipo == "bom"').pivot_table(values="tipo", index="periodo", aggfunc="count")

__Clima Perigosos__

In [ ]:
clima_perigo

__Maus Tempos__

In [ ]:
clima_mau

__Boas condições climaticas__

In [ ]:
clima_bom

__Conclusões__

Nesta analise foi possivel ser entender que as temperaturas durante o mês são aleatorias onde no principio do Mês foram altas por volta do dia 10 baixou bastante chegando em extremos. Porem no fim do Mês as temperaturas subiram novamente.
Foi possivel também por via de categorização das horas e dos tipos de clima identificar que normalmente as condições climaticas mais favoraveis estão entre as horas da madrugadas, manhas e tarde, mas têm tido tardes com climas bem perigosos e que em combinação com as noites são os periodos onde normalmente têm aparecido climas menos favoraveis.

## Análise Exploratória de Dados

#### __Quantidade de Corridas para cada Empresa de 15 a 16 de Novembro de 2017__

In [ ]:
dados_trips = df_01.sort_values(by=['trips_amount'], ascending=False).head(10)

plt.figure(figsize=(8,6))
sns.barplot(x=dados_trips['trips_amount'], y=dados_trips['company_name'], palette='viridis')
plt.xlabel('Quantidade de Entregas')
plt.ylabel('Empresa')
plt.title('Empresas com maiores quantidades de corrida de 15 a 16 de Novembro de 2017')
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.show()


#### __Bairros com mais entrega de passageiros__

In [ ]:
dados  = df_04.sort_values(by=['average_trips'], ascending=False).head(10)

plt.figure(figsize=(8, 6))
sns.barplot(x=dados['average_trips'], y=dados['dropoff_location_name'], palette='viridis')
plt.xlabel('Viagens')
plt.ylabel('local_entrega')
plt.title('Os 10 principais bairros em termos de destinos', fontsize=14)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.show()

### __Conclusões__

Neste secção de trabalho for possivel interpretar as seguintes informações:
- __Empresas com mais corridas entre 15 a 16 de Novembro__: O grafico apresentado acima ilustra que a empresa 'Flash Cab' é a empresa com mais corridas e a sua quantidade de corridas é quase o dobro da segunda classificada, a 'Taxi Affiliation Services'; As outras empresas da segunda classificada para baixo não têm muita diferença no seu numero de corridas.
- __Bairros destino com mais corridas__: O bairro 'Loop' é o bairro de destino com mais corridas a irem para Loop; do primeiro ao quarto bairro classificado é possivel verificar que as diferenças são simetricas mas do quarto para o quinto a diferença é de praticamente o dobro, mas continua simetrica do quinto para baixo.

## Teste de Hipóteses

__Indicando o nivel critico de significancia estatistica para 0.05__

In [ ]:
alpha = 0.05

__Executando teste de hipotese__

In [ ]:

duration_geral = df_07.duration_seconds

duration_baddays = df_07[df_07.weather_conditions =="Bad"].duration_seconds

results = st.ttest_ind(duration_geral, duration_baddays)


__Analizando os resultados do teste de hipotese__

In [ ]:
if results.pvalue < alpha:
    print("A duração média muda")
else:
    print("A duração média não muda")


__Conclusões__

Nesta etapa foi possivel atraves do teste de hipoteses se comprovar que a duração media dos passeios do Loop para o aeroporto Internacional O'Hare muda nos sabados quando há a presença de chuvas.

# Conclusão Geral

Este projeto de forma pratica nos permitiu perceber usando os dados como os dados na empresa Zuber mudam baseado nas condições climaticas, então algumas corridas foram afetadas por maus climas. Foi possivel se classificar os bairros de destino com mais corridas e as empresas que serviram mais corridas na plataformas.